In [7]:

import ast
from collections import Counter
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from tqdm import tqdm

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler, StandardScaler
from sklearn.linear_model import Ridge


chemin_airbnb = Path(r"C:\Users\mehdi\Documents\ProjetData\airbnb-price-prediction\data\raw\airbnb")
chemin_poi = Path(r"C:\Users\mehdi\Documents\ProjetData\airbnb-price-prediction\data\raw\poi")
chemin_sauvegarde = Path(r"C:\Users\mehdi\Documents\ProjetData\airbnb-price-prediction\data\processed")

chemin_sauvegarde.mkdir(parents=True, exist_ok=True)

In [8]:
def preparer_listings(chemin_fichier):
    df = pd.read_parquet(chemin_fichier)
    
    colonnes_a_garder = [
        'id', 'property_type', 'room_type', 'accommodates', 'bedrooms', 'beds', 'amenities',
        'latitude', 'longitude', 'neighbourhood_cleansed', 'price', 'minimum_nights', 'maximum_nights',
        'host_is_superhost', 'host_identity_verified', 'host_listings_count', 'hosts_time_as_host_years',
        'number_of_reviews', 'number_of_reviews_ltm', 'reviews_per_month', 'review_scores_rating',
        'review_scores_accuracy', 'review_scores_cleanliness', 'review_scores_checkin',
        'review_scores_communication', 'review_scores_location', 'review_scores_value',
        'availability_30', 'availability_60', 'availability_90', 'availability_365'
    ]
    
    colonnes_presentes = [c for c in colonnes_a_garder if c in df.columns]
    df = df[colonnes_presentes].copy()

    for col in ['host_is_superhost', 'host_identity_verified']:
        if col in df.columns:
            df[col] = df[col].map({'t': 1, 'f': 0, True: 1, False: 0})

    if 'price' in df.columns:
        df['price'] = df['price'].astype(str).str.replace('$', '', regex=False)
        df['price'] = pd.to_numeric(df['price'], errors='coerce')
        
        df = df.dropna(subset=['price'])
        df = df[df['price'] > 0]
        
    return df

def feature_engineering_calendrier(chemin_fichier):
    df_cal = pd.read_parquet(chemin_fichier)
    df_cal['est_dispo'] = (df_cal['available'] == 't').astype(int)
    
    stats = df_cal.groupby('listing_id').agg(
        jours_totaux=('date', 'count'),
        jours_dispos=('est_dispo', 'sum')
    ).reset_index()
    
    stats['taux_occupation'] = 1 - (stats['jours_dispos'] / stats['jours_totaux'])
    return stats[['listing_id', 'taux_occupation']]

def calcul_distance_metro(df_logements, chemin_stations):

    df_stations = pd.read_parquet(chemin_stations)
    df_geo = df_logements.dropna(subset=['latitude', 'longitude']).copy()
    
    coords_stations = df_stations['Geo Point'].str.split(',', expand=True).astype(float)
    
    rad_logements = np.radians(df_geo[['latitude', 'longitude']].values)
    rad_stations = np.radians(coords_stations.values)
    
    arbre = cKDTree(rad_stations)
    distances, index_proche = arbre.query(rad_logements, k=1)
    
    df_geo['distance_metro_m'] = np.round(distances * 6371000, 1)
    
    return df_logements.merge(df_geo[['id', 'distance_metro_m']], on='id', how='left')

In [9]:
df_listings = preparer_listings(chemin_airbnb / "listings_detailled.parquet")

df_calendrier = feature_engineering_calendrier(chemin_airbnb / "calendar.parquet")

df_final = df_listings.merge(df_calendrier, left_on='id', right_on='listing_id', how='left')
df_final = df_final.drop(columns=['listing_id'], errors='ignore')


df_final = calcul_distance_metro(df_final, chemin_poi / "train_station_scnf.parquet")


df_final.to_parquet(chemin_sauvegarde / "master_airbnb.parquet", index=False)

print(df_final.shape)

(41959, 33)


In [12]:
import pandas as pd
import numpy as np
from collections import Counter
from sklearn.base import BaseEstimator, TransformerMixin


class ExtracteurAmenities(BaseEstimator, TransformerMixin):
    def __init__(self, top_n=20):
        self.top_n = top_n

    def _convertir_en_liste(self, x):
        if pd.isna(x):
            return []
        texte_nettoye = str(x).replace('[', '').replace(']', '').replace('"', '').replace("'", "")
        return [equipement.strip() for equipement in texte_nettoye.split(',') if equipement.strip() != ""]

    def fit(self, X, y=None):
        df_temp = pd.DataFrame(X)
        if 'amenities' in df_temp.columns:
            listes = df_temp['amenities'].apply(self._convertir_en_liste)
            tous_les_items = [item for sous_liste in listes for item in sous_liste]
            # La variable se termine par "_" -> Scikit-Learn valide le fit
            self.top_amenities_ = [item for item, count in Counter(tous_les_items).most_common(self.top_n)]
        return self

    def transform(self, X):
        df_temp = pd.DataFrame(X).copy()
        if 'amenities' in df_temp.columns:
            listes = df_temp['amenities'].apply(self._convertir_en_liste)
            for am in getattr(self, 'top_amenities_', []):
                nom_colonne = f"amenity_{am.lower().replace(' ', '_').replace('/', '_')}"
                df_temp[nom_colonne] = listes.apply(lambda liste_eq: 1 if am in liste_eq else 0)
            df_temp = df_temp.drop(columns=['amenities'])
        return df_temp

class ImputerAvisAirbnb(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        df_temp = pd.DataFrame(X)
        self.medianes_ = {} # Variable requise pour valider le fit()
        
        colonnes_scores = [c for c in df_temp.columns if 'review_scores_' in str(c)]
        for col in colonnes_scores:
            self.medianes_[col] = df_temp[col].median()
        return self

    def transform(self, X):
        df_temp = pd.DataFrame(X).copy()
        
        if 'number_of_reviews' in df_temp.columns:
            df_temp['a_des_avis'] = (df_temp['number_of_reviews'] > 0).astype(int)
            
        if 'reviews_per_month' in df_temp.columns:
            df_temp['reviews_per_month'] = df_temp['reviews_per_month'].fillna(0)
            
        # Application des médianes apprises sur l'entraînement
        for col, med in getattr(self, 'medianes_', {}).items():
            if col in df_temp.columns:
                df_temp[col] = df_temp[col].fillna(med)
                
        return df_temp

class ImputerCapacite(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        df_temp = pd.DataFrame(X)
        if 'beds' in df_temp.columns and 'accommodates' in df_temp.columns:
            # Dictionnaire calculé sur l'entraînement (évite la fuite de données)
            self.lits_par_capacite_ = df_temp.groupby('accommodates')['beds'].median().to_dict()
        return self

    def transform(self, X):
        df_temp = pd.DataFrame(X).copy()
        
        if 'bedrooms' in df_temp.columns:
            df_temp['bedrooms'] = df_temp['bedrooms'].fillna(1)
            
        if 'beds' in df_temp.columns and 'accommodates' in df_temp.columns:
            # Mapping pur Pandas, avec fallback à 1
            dictionnaire_lits = getattr(self, 'lits_par_capacite_', {})
            lits_imputes = df_temp['accommodates'].map(dictionnaire_lits)
            df_temp['beds'] = df_temp['beds'].fillna(lits_imputes).fillna(1)
            
        return df_temp

In [13]:
X = df_final.drop(columns=['price', 'id'])
y = np.log1p(df_final['price'])  

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

colonnes_numeriques = ['accommodates', "bedrooms", 'beds', "minimum_nights", 'maximum_nights',
                       "number_of_reviews", 'reviews_per_month', "distance_metro_m", "taux_occupation"]
colonnes_categorielles = ['room_type', "property_type", 'neighbourhood_cleansed']

custom_transformer = Pipeline(steps=[
    ('amenities', ExtracteurAmenities(top_n=20)),
    ('reviews', ImputerAvisAirbnb()),
    ('capacite', ImputerCapacite())
])

numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('numeric', numerical_transformer, colonnes_numeriques),
    ('cat', categorical_transformer, colonnes_categorielles)
])

regressors = [
    ('Ridge', Ridge()),
    ('Random Forest', RandomForestRegressor(random_state=42, n_jobs=-1)),
    ('HistGradientBoosting', HistGradientBoostingRegressor(random_state=42))
]

params_grid = {
    'Ridge': {
        'regressor__alpha': [0.1, 1.0, 10.0], 
        'preprocessor__numeric__imputer__strategy' : ['mean', 'median']
    },
    'Random Forest': {
        'regressor__max_depth': [15, 25], 
        'preprocessor__numeric__imputer__strategy' : ['mean', 'median']
    },
    'HistGradientBoosting': {
        'regressor__learning_rate': [0.05, 0.1], 
        'regressor__max_iter': [100, 200], 
        'preprocessor__numeric__imputer__strategy' : ['mean', 'median']
    }
}

results = {}
predictions = {}

for regressor_name, regressor in tqdm(regressors, desc="Entraînement des modèles"):

    pipeline = Pipeline(steps=[
        ('custom', custom_transformer),
        ('preprocessor', preprocessor),
        ('regressor', regressor)
    ])

    model = GridSearchCV(pipeline, params_grid[regressor_name], cv=3, scoring='r2', n_jobs=-1, error_score='raise')
    
    model.fit(X_train, y_train)

    y_pred_log = model.predict(X_test)
    
    y_test_euros = np.expm1(y_test)
    y_pred_euros = np.expm1(y_pred_log)
    
    predictions[regressor_name] = y_pred_euros

    results[regressor_name] = {
        'Best params': model.best_params_, 
        'R2 on train': round(model.best_score_, 3), 
        'R2 on test': round(r2_score(y_test, y_pred_log), 3),
        'MAE in Euros': round(mean_absolute_error(y_test_euros, y_pred_euros), 2), 
        'Best model': model.best_estimator_
    }
    
df_results = pd.DataFrame(results)
df_results

Entraînement des modèles:   0%|          | 0/3 [00:00<?, ?it/s]

Entraînement des modèles: 100%|██████████| 3/3 [02:22<00:00, 47.59s/it]


,Ridge,Random Forest,HistGradientBoosting
Best params,{'preprocessor__numeric__imputer__strategy': '...,{'preprocessor__numeric__imputer__strategy': '...,{'preprocessor__numeric__imputer__strategy': '...
R2 on train,-795240.473,0.67,0.695
R2 on test,0.469,0.671,0.688
MAE in Euros,94.42,75.88,74.37
Best model,"Pipeline(steps=[('custom',\n P...","Pipeline(steps=[('custom',\n P...","Pipeline(steps=[('custom',\n P..."


In [14]:
X = df_final[[

    'accommodates',
    'bedrooms',
    'beds',
    'minimum_nights',
    'maximum_nights',
    
    'host_is_superhost',
    'host_listings_count',
    'hosts_time_as_host_years',
    

    'number_of_reviews',
    'number_of_reviews_ltm',
    'reviews_per_month',
    'review_scores_rating',
    'review_scores_accuracy',
    'review_scores_cleanliness',
    'review_scores_checkin',
    'review_scores_communication',
    'review_scores_location',
    'review_scores_value',
    

    'availability_30',
    'availability_60',
    'availability_90',
    'availability_365',
    'taux_occupation',
    'distance_metro_m',
    'latitude',
    'longitude',
    

    'room_type',
    'property_type',
    'neighbourhood_cleansed',
    
    'amenities'
]]

y = np.log1p(df_final['price'])  

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

colonnes_numeriques = [
    'accommodates',
    'bedrooms',
    'beds',
    'minimum_nights',
    'maximum_nights',
    'host_is_superhost',
    'host_listings_count',
    'hosts_time_as_host_years',
    'number_of_reviews',
    'number_of_reviews_ltm',
    'reviews_per_month',
    'review_scores_rating',
    'review_scores_accuracy',
    'review_scores_cleanliness',
    'review_scores_checkin',
    'review_scores_communication',
    'review_scores_location',
    'review_scores_value',
    'availability_30',
    'availability_60',
    'availability_90',
    'availability_365',
    'taux_occupation',
    'distance_metro_m',
]

colonnes_categorielles = [
    'room_type',
    'property_type',
    'neighbourhood_cleansed'
]
custom_transformer = Pipeline(steps=[
    ('amenities', ExtracteurAmenities(top_n=20)),
    ('reviews', ImputerAvisAirbnb()),
    ('capacite', ImputerCapacite())
])

numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('numeric', numerical_transformer, colonnes_numeriques),
    ('cat', categorical_transformer, colonnes_categorielles)
], remainder='passthrough')

X_sample = X_train.head(5)

pipeline_transformation = Pipeline(steps=[
    ('custom', custom_transformer),
    ('preprocessor', preprocessor)
])


X_transformed = pipeline_transformation.fit_transform(X_sample)


try:
    feature_names = preprocessor.get_feature_names_out()
    df_debug = pd.DataFrame(X_transformed, columns=feature_names)
except:
    df_debug = pd.DataFrame(X_transformed)

In [15]:
df = df_debug.copy()

In [16]:
df.head()

,numeric__accommodates,numeric__bedrooms,numeric__beds,numeric__minimum_nights,numeric__maximum_nights,numeric__host_is_superhost,numeric__host_listings_count,numeric__hosts_time_as_host_years,numeric__number_of_reviews,numeric__number_of_reviews_ltm,...,remainder__amenity_hangers,remainder__amenity_hair_dryer,remainder__amenity_iron,remainder__amenity_tv,remainder__amenity_coffee_maker,remainder__amenity_washer,remainder__amenity_stove,remainder__amenity_heating,remainder__amenity_dedicated_workspace,remainder__a_des_avis
0,0.0,0.0,0.0,-0.166667,2.704918,1.0,1.0,0.0,3.461538,2.333333,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0
1,-1.0,-1.0,-1.0,-0.166667,-0.786885,1.0,3.0,-0.4,0.500000,0.833333,...,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
2,0.0,1.0,0.0,0.833333,-0.885246,0.0,0.0,0.6,0.000000,-0.166667,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
3,-1.0,0.0,-1.0,4.666667,0.000000,0.0,0.0,-0.8,-0.500000,0.000000,...,1.0,1.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0
4,1.0,0.0,0.0,0.000000,0.213115,0.0,0.0,1.2,-0.538462,-0.166667,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0


In [17]:
df.describe()

,numeric__accommodates,numeric__bedrooms,numeric__beds,numeric__minimum_nights,numeric__maximum_nights,numeric__host_is_superhost,numeric__host_listings_count,numeric__hosts_time_as_host_years,numeric__number_of_reviews,numeric__number_of_reviews_ltm,...,remainder__amenity_hangers,remainder__amenity_hair_dryer,remainder__amenity_iron,remainder__amenity_tv,remainder__amenity_coffee_maker,remainder__amenity_washer,remainder__amenity_stove,remainder__amenity_heating,remainder__amenity_dedicated_workspace,remainder__a_des_avis
count,5.00000,5.000000,5.000000,5.000000,5.000000,5.000000,5.00000,5.000000,5.000000,5.000000,...,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000
mean,-0.20000,0.000000,-0.400000,1.033333,0.249180,0.400000,0.80000,0.120000,0.584615,0.566667,...,0.800000,0.800000,0.600000,0.600000,0.600000,0.600000,0.600000,0.600000,0.600000,0.800000
std,0.83666,0.707107,0.547723,2.072974,1.453822,0.547723,1.30384,0.794984,1.663078,1.071085,...,0.447214,0.447214,0.547723,0.547723,0.547723,0.547723,0.547723,0.547723,0.547723,0.447214
min,-1.00000,-1.000000,-1.000000,-0.166667,-0.885246,0.000000,0.00000,-0.800000,-0.538462,-0.166667,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,-1.00000,0.000000,-1.000000,-0.166667,-0.786885,0.000000,0.00000,-0.400000,-0.500000,-0.166667,...,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
50%,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
75%,0.00000,0.000000,0.000000,0.833333,0.213115,1.000000,1.00000,0.600000,0.500000,0.833333,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
max,1.00000,1.000000,0.000000,4.666667,2.704918,1.000000,3.00000,1.200000,3.461538,2.333333,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [18]:
X.describe()

,accommodates,bedrooms,beds,minimum_nights,maximum_nights,host_is_superhost,host_listings_count,hosts_time_as_host_years,number_of_reviews,number_of_reviews_ltm,...,review_scores_location,review_scores_value,availability_30,availability_60,availability_90,availability_365,taux_occupation,distance_metro_m,latitude,longitude
count,41959.000000,41949.000000,40306.000000,41843.000000,4.184300e+04,41959.000000,41959.000000,41959.000000,41959.000000,41959.000000,...,35216.000000,35216.000000,41959.000000,41959.000000,41959.000000,41959.000000,41959.000000,41959.000000,41959.000000,41959.000000
mean,3.298553,1.258409,1.854215,8.627775,1.543880e+05,0.305084,25.205272,5.346147,44.288734,11.311185,...,4.837111,4.621677,7.942468,23.152840,40.662885,203.046736,0.431125,269.357923,48.863623,2.341695
std,1.712824,0.860999,1.101198,29.615001,1.818313e+07,0.460448,80.205667,4.264528,84.978649,17.740539,...,0.285368,0.419357,9.287449,19.495647,30.388871,118.099128,0.317203,131.526998,0.017464,0.033568
min,1.000000,0.000000,1.000000,1.000000,1.000000e+00,0.000000,1.000000,0.000000,0.000000,0.000000,...,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.400000,48.816090,2.229896
25%,2.000000,1.000000,1.000000,1.000000,4.500000e+01,0.000000,1.000000,2.000000,2.000000,0.000000,...,4.800000,4.500000,0.000000,4.000000,10.000000,89.000000,0.140921,175.100000,48.851220,2.320085
50%,3.000000,1.000000,2.000000,2.000000,3.650000e+02,0.000000,2.000000,4.000000,15.000000,4.000000,...,4.920000,4.710000,4.000000,20.000000,42.000000,233.000000,0.345205,254.700000,48.864860,2.346010
75%,4.000000,2.000000,2.000000,4.000000,6.000000e+02,1.000000,10.000000,9.000000,50.000000,15.000000,...,5.000000,4.860000,12.000000,38.000000,66.000000,312.000000,0.734247,349.800000,48.876870,2.364690
max,16.000000,33.000000,30.000000,365.000000,2.147484e+09,1.000000,3290.000000,15.000000,4760.000000,711.000000,...,5.000000,5.000000,30.000000,60.000000,90.000000,365.000000,0.997260,1618.800000,48.901590,2.468380


In [19]:
print(X['maximum_nights'].min(), X['maximum_nights'].max())

1.0 2147483647.0


In [20]:
X['maximum_nights'].unique()

array([3.00000000e+01, 1.30000000e+02, 3.10000000e+01, 9.00000000e+01,
       3.05000000e+02, 1.12500000e+03, 2.10000000e+01, 1.20000000e+02,
       7.30000000e+02, 7.20000000e+01, 3.00000000e+02, 4.00000000e+01,
       1.80000000e+02, 7.00000000e+00, 2.80000000e+01, 1.50000000e+01,
       1.40000000e+01, 2.00000000e+01, 3.65000000e+02, 6.00000000e+01,
       1.20000000e+01, 3.50000000e+01, 1.12400000e+03, 2.50000000e+01,
       1.80000000e+01, 4.50000000e+01, 4.00000000e+02, 5.00000000e+00,
       1.00000000e+01, 3.06000000e+02, 7.29000000e+02, 7.50000000e+01,
       6.50000000e+01, 4.00000000e+00, 7.00000000e+02, 3.03000000e+02,
       5.00000000e+01, 3.50000000e+02, 6.00000000e+02, 6.00000000e+00,
       1.60000000e+01, 2.90000000e+01, 1.60000000e+02, 1.00000000e+03,
       2.10000000e+02, 3.00000000e+00, 2.20000000e+01, 3.20000000e+01,
       6.20000000e+01, 1.70000000e+01, 3.60000000e+02, 1.19000000e+02,
       1.10000000e+01, 1.10000000e+02, 1.90000000e+01, 2.40000000e+02,
      

In [21]:
X.loc[X['maximum_nights'] > 365, 'maximum_nights'] = 365

In [22]:
print(X['maximum_nights'].min(), X['maximum_nights'].max())

1.0 365.0


In [23]:
X['maximum_nights'].unique()

array([ 30., 130.,  31.,  90., 305., 365.,  21., 120.,  72., 300.,  40.,
       180.,   7.,  28.,  15.,  14.,  20.,  60.,  12.,  35.,  25.,  18.,
        45.,   5.,  10., 306.,  75.,  65.,   4., 303.,  50., 350.,   6.,
        16.,  29., 160., 210.,   3.,  22.,  32.,  62.,  17., 360., 119.,
        11., 110.,  19., 240., 100.,   8., 121.,  34.,  92., 200.,   9.,
       352.,  33., 310.,   2.,  80., 150.,  24.,  70., 270., 125., 364.,
        13.,  66.,  99.,  93., 330., 112.,  27., 183.,   1.,  57., 250.,
        48.,  39.,  23., 243.,  95.,  56., 140., 230., 336.,  89., 304.,
       356.,  49., 290., 260.,  91., 151.,  26.,  84.,  42.,  63.,  37.,
        41.,  69., 109., 308.,  61., 117., 340.,  67.,  85., 275., 118.,
       320.,  76., 311.,  55., 182., 301., 184.,  94., 190.,  73., 235.,
        38., 225., 355., 128., 325., 299., 280.,  81., 232., 122.,  98.,
       239., 334., 155., 274., 124.,  59., 220., 307., 214.,  52.,  36.,
       113., 185., 245., 115.,  nan, 345.,  71., 24

In [24]:
class NettoyeurMaximumNights(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        df_temp = pd.DataFrame(X).copy()
        if 'maximum_nights' in df_temp.columns:
            df_temp.loc[df_temp['maximum_nights'] > 365, 'maximum_nights'] = 365
        return df_temp

In [25]:
X = df_final[[

    'accommodates',
    'bedrooms',
    'beds',
    'minimum_nights',
    'maximum_nights',
    
    'host_is_superhost',
    'host_listings_count',
    'hosts_time_as_host_years',
    

    'number_of_reviews',
    'number_of_reviews_ltm',
    'reviews_per_month',
    'review_scores_rating',
    'review_scores_accuracy',
    'review_scores_cleanliness',
    'review_scores_checkin',
    'review_scores_communication',
    'review_scores_location',
    'review_scores_value',
    

    'availability_30',
    'availability_60',
    'availability_90',
    'availability_365',
    'taux_occupation',
    'distance_metro_m',
    'latitude',
    'longitude',
    

    'room_type',
    'property_type',
    'neighbourhood_cleansed',
    
    'amenities'
]]

y = np.log1p(df_final['price'])  

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

colonnes_numeriques = [
    'accommodates',
    'bedrooms',
    'beds',
    'minimum_nights',
    'maximum_nights',
    'host_is_superhost',
    'host_listings_count',
    'hosts_time_as_host_years',
    'number_of_reviews',
    'number_of_reviews_ltm',
    'reviews_per_month',
    'review_scores_rating',
    'review_scores_accuracy',
    'review_scores_cleanliness',
    'review_scores_checkin',
    'review_scores_communication',
    'review_scores_location',
    'review_scores_value',
    'availability_30',
    'availability_60',
    'availability_90',
    'availability_365',
    'taux_occupation',
    'distance_metro_m',
]

colonnes_categorielles = [
    'room_type',
    'property_type',
    'neighbourhood_cleansed'
]

custom_transformer = Pipeline(steps=[
    ('nettoyage_nuits', NettoyeurMaximumNights()),
    ('amenities', ExtracteurAmenities(top_n=20)),
    ('reviews', ImputerAvisAirbnb()),
    ('capacite', ImputerCapacite())
])

numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Correction : On applique un scaler sur le remainder pour scalériser les colonnes des amenities
preprocessor = ColumnTransformer(transformers=[
    ('numeric', numerical_transformer, colonnes_numeriques),
    ('cat', categorical_transformer, colonnes_categorielles)
])

regressors = [
    ('Ridge', Ridge()),
    ('Random Forest', RandomForestRegressor(random_state=42, n_jobs=-1)),
    ('HistGradientBoosting', HistGradientBoostingRegressor(random_state=42))
]

params_grid = {
    'Ridge': {
        'regressor__alpha': [0.1, 1.0, 10.0], 
        'preprocessor__numeric__imputer__strategy' : ['mean', 'median']
    },
    'Random Forest': {
        'regressor__max_depth': [15, 25], 
        'preprocessor__numeric__imputer__strategy' : ['mean', 'median']
    },
    'HistGradientBoosting': {
        'regressor__learning_rate': [0.05, 0.1], 
        'regressor__max_iter': [100, 200], 
        'preprocessor__numeric__imputer__strategy' : ['mean', 'median']
    }
}

results = {}
predictions = {}

for regressor_name, regressor in tqdm(regressors, desc="Entraînement des modèles"):

    pipeline = Pipeline(steps=[
        ('custom', custom_transformer),
        ('preprocessor', preprocessor),
        ('regressor', regressor)
    ])

    model = GridSearchCV(pipeline, params_grid[regressor_name], cv=3, scoring='r2', n_jobs=-1, error_score='raise')
    
    model.fit(X_train, y_train)

    y_pred_log = model.predict(X_test)
    
    y_test_euros = np.expm1(y_test)
    y_pred_euros = np.expm1(y_pred_log)
    
    predictions[regressor_name] = y_pred_euros

    results[regressor_name] = {
        'Best params': model.best_params_, 
        'R2 on train': round(model.best_score_, 3), 
        'R2 on test': round(r2_score(y_test, y_pred_log), 3),
        'MAE in Euros': round(mean_absolute_error(y_test_euros, y_pred_euros), 2), 
        'Best model': model.best_estimator_
    }
    
df_results = pd.DataFrame(results)
df_results

Entraînement des modèles: 100%|██████████| 3/3 [03:13<00:00, 64.43s/it]


,Ridge,Random Forest,HistGradientBoosting
Best params,{'preprocessor__numeric__imputer__strategy': '...,{'preprocessor__numeric__imputer__strategy': '...,{'preprocessor__numeric__imputer__strategy': '...
R2 on train,0.551,0.708,0.748
R2 on test,0.561,0.719,0.748
MAE in Euros,87.75,70.72,66.92
Best model,"Pipeline(steps=[('custom',\n P...","Pipeline(steps=[('custom',\n P...","Pipeline(steps=[('custom',\n P..."


In [26]:
X = df_final[[

    'accommodates',
    'bedrooms',
    'beds',
    'minimum_nights',
    'maximum_nights',
    
    'host_is_superhost',
    'host_listings_count',
    'hosts_time_as_host_years',
    

    'number_of_reviews',
    'number_of_reviews_ltm',
    'reviews_per_month',
    'review_scores_rating',
    'review_scores_accuracy',
    'review_scores_cleanliness',
    'review_scores_checkin',
    'review_scores_communication',
    'review_scores_location',
    'review_scores_value',
    

    'availability_30',
    'availability_60',
    'availability_90',
    'availability_365',
    'taux_occupation',
    'distance_metro_m',
    'latitude',
    'longitude',
    

    'room_type',
    'property_type',
    'neighbourhood_cleansed',
    
    'amenities'
]]

y = np.log1p(df_final['price'])  

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

colonnes_numeriques = [
    'accommodates',
    'bedrooms',
    'beds',
    'minimum_nights',
    'maximum_nights',
    'host_is_superhost',
    'host_listings_count',
    'hosts_time_as_host_years',
    'number_of_reviews',
    'number_of_reviews_ltm',
    'reviews_per_month',
    'review_scores_rating',
    'review_scores_accuracy',
    'review_scores_cleanliness',
    'review_scores_checkin',
    'review_scores_communication',
    'review_scores_location',
    'review_scores_value',
    'availability_30',
    'availability_60',
    'availability_90',
    'availability_365',
    'taux_occupation',
    'distance_metro_m',
]

colonnes_categorielles = [
    'room_type',
    'property_type',
    'neighbourhood_cleansed'
]

custom_transformer = Pipeline(steps=[
    ('nettoyage_nuits', NettoyeurMaximumNights()),
    ('amenities', ExtracteurAmenities(top_n=20)),
    ('reviews', ImputerAvisAirbnb()),
    ('capacite', ImputerCapacite())
])

numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False, drop='first'))
])

preprocessor = ColumnTransformer(transformers=[
    ('numeric', numerical_transformer, colonnes_numeriques),
    ('cat', categorical_transformer, colonnes_categorielles)
]) 

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor

regressors = [
    ('Ridge', Ridge()),
    ('Random Forest', RandomForestRegressor(random_state=42, n_jobs=-1)),
    ('HistGradientBoosting', HistGradientBoostingRegressor(random_state=42)),
    ('SVR', SVR()),
    ('KNN', KNeighborsRegressor(n_jobs=-1))
]

params_grid = {
    'Ridge': {
        'regressor__alpha': [0.1, 1.0, 10.0, 100.0]
    },
    'Random Forest': {
        'regressor__n_estimators': [100, 200],
        'regressor__max_depth': [15, 25]
    },
    'HistGradientBoosting': {
        'regressor__learning_rate': [0.05, 0.1],
        'regressor__max_iter': [100, 200]
    },
    'SVR': {
        'regressor__C': [1.0, 10.0],
        'regressor__epsilon': [0.1, 0.2]
    },
    'KNN': {
        'regressor__n_neighbors': [5, 10, 15]
    }
}

results = {}
predictions = {}

for regressor_name, regressor in tqdm(regressors, desc="Entraînement des modèles"):

    pipeline = Pipeline(steps=[
        ('custom', custom_transformer),
        ('preprocessor', preprocessor),
        ('regressor', regressor)
    ])

    model = GridSearchCV(
        pipeline, 
        params_grid[regressor_name], 
        cv=3, 
        scoring='r2', 
        n_jobs=-1, 
        error_score='raise'
    )
    
    model.fit(X_train, y_train)

    y_pred_log = model.predict(X_test)
    
    y_test_euros = np.expm1(y_test)
    y_pred_euros = np.expm1(y_pred_log)
    
    predictions[regressor_name] = y_pred_euros

    results[regressor_name] = {
        'Best params': model.best_params_, 
        'R2 on train': round(r2_score(y_train, model.best_estimator_.predict(X_train)), 3),
        'R2 CV': round(model.best_score_, 3),
        'R2 on test': round(r2_score(y_test, y_pred_log), 3),
        'MAE in Euros': round(mean_absolute_error(y_test_euros, y_pred_euros), 2), 
        'Best model': model.best_estimator_
    }
    
df_results = pd.DataFrame(results)
df_results

Entraînement des modèles:   0%|          | 0/5 [00:00<?, ?it/s]c:\Users\mehdi\Documents\ProjetData\airbnb-price-prediction\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
Entraînement des modèles:  20%|██        | 1/5 [00:09<00:39,  9.98s/it]c:\Users\mehdi\Documents\ProjetData\airbnb-price-prediction\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
Entraînement des modèles:  40%|████      | 2/5 [03:27<06:00, 120.19s/it]c:\Users\mehdi\Documents\ProjetData\airbnb-price-prediction\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be enc

,Ridge,Random Forest,HistGradientBoosting,SVR,KNN
Best params,{'regressor__alpha': 1.0},"{'regressor__max_depth': 25, 'regressor__n_est...","{'regressor__learning_rate': 0.1, 'regressor__...","{'regressor__C': 10.0, 'regressor__epsilon': 0.2}",{'regressor__n_neighbors': 15}
R2 on train,0.555,0.951,0.802,0.761,0.694
R2 CV,0.551,0.71,0.747,0.726,0.632
R2 on test,0.561,0.721,0.749,0.728,0.649
MAE in Euros,87.74,70.57,66.69,68.75,77.2
Best model,"Pipeline(steps=[('custom',\n P...","Pipeline(steps=[('custom',\n P...","Pipeline(steps=[('custom',\n P...","Pipeline(steps=[('custom',\n P...","Pipeline(steps=[('custom',\n P..."
